In [ ]:
import requests
import os
from dotenv import load_dotenv
from modules.get_token import get_token
load_dotenv()
start_url = os.getenv("PRJ_START_URL")

In [ ]:
import json


def get_invoice_schema(token):
    """
    Fetch all available fields for the Invoice type using GraphQL introspection
    """
    url = f"{start_url}/graphql?token={token}"
    headers = {"content-type": "application/json"}
    
    # GraphQL introspection query to get Invoice type fields
    introspection_query = """
    query IntrospectionQuery {
      __type(name: "Invoice") {
        name
        description
        fields(includeDeprecated: false) {
          name
          description
          type {
            name
            kind
            ofType {
              name
              kind
            }
          }
        }
      }
    }
    """
    
    payload = {"query": introspection_query}
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        if 'errors' in data:
            print("GraphQL Errors:")
            print(json.dumps(data['errors'], indent=2))
            return None
            
        invoice_type = data.get('data', {}).get('__type')
        
        if not invoice_type:
            print("Invoice type not found in schema")
            return None
            
        return invoice_type
        
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None


def display_fields(invoice_type):
    """
    Display all available fields in a readable format
    """
    if not invoice_type:
        return
        
    print(f"\n{'='*80}")
    print(f"Type: {invoice_type['name']}")
    if invoice_type.get('description'):
        print(f"Description: {invoice_type['description']}")
    print(f"{'='*80}\n")
    
    fields = invoice_type.get('fields', [])
    
    print(f"Total Available Fields: {len(fields)}\n")
    print(f"{'Field Name':<30} {'Type':<25} {'Description'}")
    print(f"{'-'*30} {'-'*25} {'-'*40}")
    
    for field in fields:
        field_name = field['name']
        field_type = field['type']
        
        # Parse type information
        if field_type['kind'] == 'NON_NULL':
            type_name = f"{field_type['ofType']['name']}!"
        elif field_type['kind'] == 'LIST':
            type_name = f"[{field_type['ofType']['name']}]"
        else:
            type_name = field_type.get('name', 'Unknown')
        
        description = field.get('description') or ''
        description_display = description[:40] if description else ''
        
        print(f"{field_name:<30} {type_name:<25} {description_display}")
    
    return fields


def generate_sample_query(fields):
    """
    Generate a sample GraphQL query with all available fields
    """
    if not fields:
        return
        
    print(f"\n{'='*80}")
    print("SAMPLE QUERY WITH ALL FIELDS:")
    print(f"{'='*80}\n")
    
    # Filter out complex nested types for the basic query
    simple_fields = []
    for field in fields:
        field_type = field['type']
        type_kind = field_type.get('kind', '')
        
        # Include scalar types and enums
        if type_kind in ['SCALAR', 'ENUM', 'NON_NULL']:
            simple_fields.append(field['name'])
        elif type_kind == 'OBJECT':
            # Mark complex objects
            simple_fields.append(f"{field['name']} # OBJECT - expand as needed")
        elif type_kind == 'LIST':
            simple_fields.append(f"{field['name']} # LIST - expand as needed")
    
    query = '''query invoice($invoiceId: String!) {
    invoice(invoiceId: $invoiceId) {
'''
    
    for field_name in simple_fields:
        query += f"        {field_name}\n"
    
    query += '''    }
}'''
    
    print(query)
    print()


def save_to_file(invoice_type, filename="invoice_fields.json"):
    """
    Save the schema information to a JSON file
    """
    try:
        with open(filename, 'w') as f:
            json.dump(invoice_type, f, indent=2)
        print(f"\n✓ Schema saved to {filename}")
    except Exception as e:
        print(f"\n✗ Failed to save schema: {e}")


def main():
    print("Fetching Invoice schema from GraphQL API...")
    print("="*80)
    
    token = get_token()
    if not token:
        print("Failed to get authentication token")
        return
    
    # Get the schema
    invoice_type = get_invoice_schema(token)
    
    # Display fields
    fields = display_fields(invoice_type)
    
    # Generate sample query
    generate_sample_query(fields)
    
    # Save to file
    save_to_file(invoice_type)
    
    print(f"\n{'='*80}")
    print("CURRENTLY USED FIELDS IN YOUR SCRIPT:")
    print(f"{'='*80}")
    current_fields = [
        'invoiceId',
        'invoiceDate',
        'invoiceDueDate',
        'clientPONum',
        'clientPartNumber',
        'invoicedDollars',
        'status'
    ]
    for field in current_fields:
        print(f"  • {field}")
    
    print("\n✓ Compare with the full list above to see what else is available!")


if __name__ == "__main__":
    main()

Fetching Invoice schema from GraphQL API...

Type: Invoice

Total Available Fields: 64

Field Name                     Type                      Description
------------------------------ ------------------------- ----------------------------------------
bulkImportId                   String                    Bulk Import Id
clientPartNumber               String                    
clientPoNum                    String                    
copiedFrom                     String                    ID of record from which this record was 
copiedTime                     String                    Timestamp when this record was copied
createdTime                    String                    Timestamp when this record was created
currency                       String                    Payment Currency
The currency in which t
currencyLocale                 String                    Localized Currency Format
customerPO                     CustomerPO                Customer PO #
customerPOPlainT